# Battle Tailscale Worker（单 cell）

Tailscale 内网直连，替代 cloudflared。**改 CFG 后点 Run 即就绪。**

| mode | rl_mode | 说明 | 本地要做什么 |
|---|---|---|---|
| `rl` | `pull` | Colab 轮询本地 hub | hub 跑在本地（Tailscale IP，或同机 `127.0.0.1:8787`） |
| `rl` | `push` | Colab 起 worker，本地推 job | rl-config 节点 url → `http://<TS_IP>:<push_port>` |
| `bc` | — | Colab 本地 BC 蒸馏 | 上传语料 zip |

**代码住哪**（改运行时逻辑优先改仓库，不必重发 notebook）：

| 部分 | 住哪 |
|---|---|
| CFG / 凭据 / 保活 / 拉远端模块 / 内联回退 | 本 cell |
| pull / push / bc 分支 | `remote/notebook_boot.py` |
| Tailscale 安装·起 daemon·登录·取 IP·诊断 | `remote/tailscale_boot.py` |
| 设备探测 / 守候 / 监督 / 完整 worker | `remote/notebook_runtime.py`、`remote/push_bootstrap.py`（code.zip） |

前两个从 GitHub raw 拉（`repo_url` + `branch`）；**拉不到时 cell 走内联回退，
只支持 `rl/pull`**（push、bc 需要远端可达）。

**凭据**：一律留空。取用顺序 = 环境变量 → Colab/Kaggle Secrets → CFG 手填；
键名 `TS_AUTHKEY` / `HUB_TOKEN` / `PUSH_TOKEN`，**值永不进日志**。

**连不上**时用 `ipynb/tailscale.debug.ipynb` 分层体检（daemon/引擎/登录/peer/hub 连通）。

**停止**：■ 中断本 cell。


In [ ]:
# @title Battle Tailscale Worker —— 改参数后点 Run
# 本 cell 只留：CFG / 凭据 / 保活 / 拉远端引导模块 / 内联回退。
# 其余在 remote/notebook_boot.py + remote/tailscale_boot.py（GitHub raw）。
import io, os, shutil, subprocess, sys, threading, time, urllib.error, urllib.request, zipfile
from pathlib import Path

CFG = {
    "mode": "rl",                 # rl | bc
    "rl_mode": "pull",            # push | pull
    # ── 凭据一律留空：环境变量 → Colab/Kaggle Secrets → 这里手填 ──
    "ts_authkey": "",             # TS_AUTHKEY
    "ts_ephemeral": True,         # 会话结束自动摘节点（1.10x 起取决于 key 本身是否 ephemeral）
    "push_port": 8790,
    "push_token": "",             # PUSH_TOKEN（push 必填）
    "hub_url": "http://<本地TS_IP>:8787",   # pull：本地 hub（同机可直接 http://127.0.0.1:8787）
    "hub_token": "",              # HUB_TOKEN
    "device": "auto",
    "max_session_hours": 9,
    "repo_url": "https://github.com/HuangJian/battle.git",
    "branch": "goal-nn",
    # ── BC（回退模式不支持）──
    "bc_run_tag": "p3bc",
    "bc_course": "p3-bc",
    "bc_corpus_zip": "p3-godai.zip",
    "bc_epochs": 60,
    "bc_seed": 1234,
}


def _log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] [battle] {msg}", flush=True)


def _secret(key, cfg_val=""):
    """环境变量 → Colab/Kaggle Secrets → CFG 手填（值永不进日志）。"""
    def _env():
        return os.environ.get(key, "")

    def _colab():
        from google.colab import userdata
        return userdata.get(key) or ""

    def _kaggle():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key) or ""

    for _get in (_env, _colab, _kaggle):
        try:
            _v = _get()
        except Exception:
            _v = ""
        if _v:
            return str(_v).strip()
    return str(cfg_val or "").strip()


_log(f"mode={CFG['mode']}" + (f"/{CFG['rl_mode']}" if CFG["mode"] == "rl" else ""))

# ── Keepalive ──────────────────────────────────────────────
_keepalive_stop = threading.Event()


def _keepalive_loop():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
        _sentinel = "/tmp/battle-halt-request"
        while not _keepalive_stop.is_set():
            # 停机哨兵：worker 子进程写入后，由本 kernel 执行 unassign（子进程做不到）
            if os.path.exists(_sentinel):
                print(f"[{time.strftime('%H:%M:%S')}] [keepalive] 检测到停机哨兵 → 释放 Colab 实例", flush=True)
                try:
                    from google.colab import runtime
                    runtime.unassign()
                except Exception as _e:
                    print(f"[{time.strftime('%H:%M:%S')}] [keepalive] unassign 失败: {_e}——请手工断开", flush=True)
                break
            try:
                from IPython.display import Javascript, display
                display(Javascript(
                    "function f(){document.querySelector('colab-connect-button')?.click();}"
                    "setTimeout(f,1000);"))
            except Exception:
                pass
            _keepalive_stop.wait(60)
    else:
        n = 0
        while not _keepalive_stop.is_set():
            print(f"[{time.strftime('%H:%M:%S')}] [keepalive] alive ({n * 3} min)", flush=True)
            n += 1
            _keepalive_stop.wait(180)


threading.Thread(target=_keepalive_loop, daemon=True, name="keepalive").start()
_log("Keepalive 已启动")


def _load_boot(log):
    """从 GitHub raw 取 remote/{notebook_boot,tailscale_boot}.py（有缓存先用缓存）。"""
    dst_dir = Path("/tmp/battle-boot")
    base = CFG["repo_url"].replace("github.com", "raw.githubusercontent.com").rstrip("/")
    if base.endswith(".git"):
        base = base[:-4]
    _marker = dst_dir / "_branch.txt"
    if _marker.exists() and _marker.read_text(encoding="utf-8").strip() != CFG["branch"]:
        for _f in ("notebook_boot.py", "tailscale_boot.py"):
            (dst_dir / _f).unlink(missing_ok=True)
    dst_dir.mkdir(parents=True, exist_ok=True)
    _marker.write_text(CFG["branch"], encoding="utf-8")
    for _name in ("notebook_boot.py", "tailscale_boot.py"):
        _dst = dst_dir / _name
        if _dst.exists():
            continue
        try:
            with urllib.request.urlopen(
                f"{base}/{CFG['branch']}/nn-training/remote/{_name}", timeout=30) as _r:
                _data = _r.read()
            if b"def " not in _data:
                raise ValueError("内容不像 Python 源文件")
            dst_dir.mkdir(parents=True, exist_ok=True)
            _dst.write_bytes(_data)
        except Exception as _e:
            log(f"拉 {_name} 失败（{type(_e).__name__}: {_e}）——回退内联精简版")
            return None
    sys.path.insert(0, str(dst_dir))
    try:
        import notebook_boot
    except Exception as _e:
        log(f"导入引导模块失败（{type(_e).__name__}: {_e}）——回退内联精简版")
        return None
    log(f"引导模块已载入（{CFG['branch']}）")
    return notebook_boot


_boot = _load_boot(_log)
if _boot is not None:
    raise SystemExit(_boot.run(CFG, _log, _secret, _keepalive_stop))

# ══════════════════════════════════════════════════════════
# 内联回退：远端模块不可用时的最小可用路径（Tailscale + pull）
# ══════════════════════════════════════════════════════════
if CFG["mode"] != "rl" or str(CFG.get("rl_mode") or "push").lower() != "pull":
    raise SystemExit(
        "[FATAL] 远端引导模块不可用；回退版只支持 rl/pull。"
        "（push / bc 需要 repo_url+branch 可从本机访问）")

_SOCK, _STATE, _DLOG, _PROXY = (
    "/var/run/tailscale/tailscaled.sock",
    "/tmp/tailscale-state",
    "/tmp/tailscaled.log",
    "localhost:1055",
)


def _ts(*args, timeout=30):
    return subprocess.run(
        ["tailscale", f"--socket={_SOCK}", *args],
        capture_output=True, text=True, timeout=timeout)


def _inline_ensure(authkey, log):
    """只走 userspace 一条路（Colab 实测 kernel 模式 rc=1 秒退）。"""
    if not shutil.which("tailscale"):
        log("安装 Tailscale …")
        subprocess.run("curl -fsSL https://tailscale.com/install.sh | sh",
                       shell=True, check=True, timeout=180)
    if not Path(_SOCK).exists():
        Path(_SOCK).parent.mkdir(parents=True, exist_ok=True)
        Path(_STATE).mkdir(parents=True, exist_ok=True)
        subprocess.Popen(
            ["tailscaled", "--tun=userspace-networking", f"--socks5-server={_PROXY}",
             f"--outbound-http-proxy-listen={_PROXY}", f"--socket={_SOCK}",
             f"--statedir={_STATE}"],
            stdout=open(_DLOG, "ab"), stderr=subprocess.STDOUT, start_new_session=True)
        for _ in range(30):
            if Path(_SOCK).exists():
                break
            time.sleep(1)
    _r = _ts(*(["up"] + ([f"--auth-key={authkey}"] if authkey else [])), timeout=120)
    if _r.returncode != 0:
        log(((_r.stdout or "") + (_r.stderr or "")).strip()[:1500])
        raise RuntimeError("tailscale up 失败（key 用过/过期？看上一行首句）")
    ip = ""
    for _ in range(30):
        _r = _ts("ip", "-4")
        if _r.returncode == 0 and _r.stdout.strip():
            ip = _r.stdout.strip()
            break
        time.sleep(1)
    if not ip:
        raise RuntimeError("Tailscale 未能获取 IP")
    for _k, _v in (("HTTP_PROXY", f"http://{_PROXY}"), ("ALL_PROXY", f"socks5://{_PROXY}"),
                   ("NO_PROXY", "localhost,127.0.0.1")):
        os.environ[_k] = _v
        os.environ[_k.lower()] = _v
    log(f"Tailscale IP = {ip} (mode=userspace·内联回退)")
    return ip


_ak = _secret("TS_AUTHKEY", CFG.get("ts_authkey"))
if not _ak:
    _log("!! 未拿到 TS_AUTHKEY（环境变量 / Colab Secrets / CFG 三处都没有）")
_hub = str(CFG.get("hub_url") or "").strip()
_hub_tok = _secret("HUB_TOKEN", CFG.get("hub_token"))
if not _hub:
    raise SystemExit("[FATAL] pull 需要 hub_url")

_inline_ensure(_ak, _log)   # 必须在拉 hub 之前：userspace 模式靠它注入出站代理

# Colab 的 urlopen() 不读 HTTP_PROXY 环境变量（2026-09-16 实测）——必须显式 ProxyHandler
_proxies = {}
for _k in ("http_proxy", "HTTP_PROXY"):
    _v = os.environ.get(_k)
    if _v: _proxies["http"] = _v; break
for _k in ("https_proxy", "HTTPS_PROXY"):
    _v = os.environ.get(_k)
    if _v: _proxies["https"] = _v; break
_opener = urllib.request.build_opener(urllib.request.ProxyHandler(_proxies)) if _proxies else urllib.request.build_opener()

_deadline = time.time() + 3600
while True:
    try:
        _req = urllib.request.Request(_hub.rstrip("/") + "/code",
                                      headers={"Authorization": "Bearer " + _hub_tok})
        with _opener.open(_req, timeout=120) as _resp:
            _raw = _resp.read()
        _log(f"code.zip 就绪: {len(_raw)} bytes")
        break
    except urllib.error.HTTPError as _e:
        if _e.code in (401, 403):
            raise SystemExit(f"[FATAL] /code HTTP {_e.code} — HUB_TOKEN 不一致") from None
        _log(f"/code HTTP {_e.code} —— 30s 后重试")
        time.sleep(30)
    except Exception as _e:
        if time.time() > _deadline:
            raise SystemExit("[FATAL] 等 code.zip 超 1h") from None
        _log(f"hub 异常（{type(_e).__name__}）—— 30s 重试")
        time.sleep(30)

_code_dir = "/tmp/worker-code"
Path(_code_dir).mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(_raw)) as _z:
    _z.extractall(_code_dir)
sys.path.insert(0, _code_dir)
from remote.notebook_runtime import run_notebook
raise SystemExit(run_notebook({
    "mode": "pull", "hub_url": _hub, "hub_token": _hub_tok,
    "push_port": int(CFG["push_port"]),
    "push_token": _secret("PUSH_TOKEN", CFG.get("push_token")),
    "device": CFG["device"], "use_multi_gpu": True,
    "max_session_hours": CFG["max_session_hours"], "poll_interval_sec": 1,
    "idle_floor_sec": 3600, "max_worker_restarts": 5,
    "keepalive_stop": _keepalive_stop, "log": _log, "code_dir": _code_dir,
}))
